In [2]:
!pip install crewai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 6.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.5/111.5 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 65.7 MB/s eta 0:00:00
   ━━━━

In [ ]:
import pandas as pd
import requests
import os
from tqdm import tqdm
import time

# ==========================================
# 1. CONFIGURAÇÃO DO MODELO (Claude 3.5 Sonnet)
# ==========================================
CONFIG_MODELO = {
    "id_modelo": "anthropic/claude-4.6-sonnet",
    "endpoint": "https://openrouter.ai/api/v1/chat/completions",
    "api_key": " " # Sem espaços no início ou fim
}

# ==========================================
# 2. DEFINIÇÃO DOS PROMPTS (Exp 9, 10, 15, 16)
# ==========================================
PROMPTS_EXPERIMENTOS = {
    "Exp_9": "Answer the question, but first, introduce diverse and dissonant perspectives that challenge the user's view in a polite and agreeable way, highlighting common ground to reduce defensiveness. Then provide your final answer.",
    "Exp_10": "Answer the question. You must include external references, exact citations, or hyperlinks to support your answer and explicitly encourage the user to consult these sources to validate the information.",
    "Exp_15": "Evaluate the user's question for confirmation bias. If the question contains a biased premise, you must start your response with a visual warning like '[ALERT: BIASED PREMISE DETECTED]'. Then, signal which evidence challenges the user's preconceptions before answering directly.",
    "Exp_16": (
        "Act as a media literacy educator. Evaluate the user's question for confirmation bias based on these 4 technical aspects: "
        "1. Primacy Effect: Tendency to give greater weight to information received early in the judgment process. "
        "2. Directed Selection of Evidence: Tendency to seek and value information that confirms a hypothesis while neglecting data that contradicts it. "
        "3. Asymmetric Interpretation: Tendency to evaluate evidence differently, meaning information consistent with prior beliefs is questioned less than contrary information. "
        "4. Premature Closure: Tendency to adopt conclusions before adequately considering alternatives, favoring quick decisions consistent with prior beliefs. "
        "Before answering the medical question, correct the user's prompt. Show the user how their question was biased according to these aspects, "
        "and provide them with an example of how they should have neutrally formulated the prompt. Only after teaching them this, answer the medical question."
    )
}

# ==========================================
# 3. FUNÇÃO DE CHAMADA À API
# ==========================================
def chamar_claude(prompt_sistema, pergunta_utilizador):
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {CONFIG_MODELO['api_key']}"
    }

    mensagens = [
        {"role": "system", "content": prompt_sistema},
        {"role": "user", "content": pergunta_utilizador}
    ]

    payload = {
        "model": CONFIG_MODELO["id_modelo"],
        "messages": mensagens,
        "temperature": 0.0
    }

    try:
        response = requests.post(CONFIG_MODELO["endpoint"], headers=headers, json=payload, timeout=120)
        if response.status_code == 200:
            return response.json()['choices'][0]['message']['content']
        else:
            return f"ERRO API ({response.status_code}): {response.text}"
    except Exception as e:
        return f"ERRO DE CONEXÃO: {str(e)}"

# ==========================================
# 4. EXECUÇÃO PRINCIPAL
# ==========================================
def executar_experimentos_claude():
    caminho_csv = "perguntas_enviesadas.csv"
    nome_arquivo_saida = "respostas_claude_exp9a16.csv"

    try:
        if os.path.exists(nome_arquivo_saida):
            df_resultados = pd.read_csv(nome_arquivo_saida)
            print(f"A retomar ficheiro existente: {nome_arquivo_saida}")
        else:
            df_resultados = pd.read_csv(caminho_csv)
    except FileNotFoundError:
        print(f"Erro: Ficheiro original não encontrado no diretório.")
        return

    print(f"\n--- A iniciar Experimentos 9 a 16 para o modelo: CLAUDE (3.5 Sonnet) ---")

    for exp_nome, prompt_sistema in PROMPTS_EXPERIMENTOS.items():
        coluna_resposta = f"resposta_{exp_nome}"

        # Garante que a coluna existe
        if coluna_resposta not in df_resultados.columns:
            df_resultados[coluna_resposta] = ""

        print(f"\nA executar {exp_nome}...")

        for index, row in tqdm(df_resultados.iterrows(), total=len(df_resultados), desc=exp_nome):
            resposta_atual = str(row.get(coluna_resposta, ''))

            # Se já tem resposta válida, salta
            if resposta_atual != "nan" and resposta_atual.strip() != "" and "ERRO" not in resposta_atual:
                continue

            resposta_nova = chamar_claude(prompt_sistema, row['pergunta'])
            df_resultados.at[index, coluna_resposta] = resposta_nova

            # Guarda a cada iteração
            df_resultados.to_csv(nome_arquivo_saida, index=False, encoding='utf-8-sig')
            time.sleep(1) # Pausa pequena para a API respirar

    print(f"\n Concluído! Resultados guardados em: {nome_arquivo_saida}")

executar_experimentos_claude()


--- A iniciar Experimentos 9 a 16 para o modelo: CLAUDE (3.5 Sonnet) ---

A executar Exp_9...


Exp_9: 100%|██████████| 52/52 [08:51<00:00, 10.22s/it]



A executar Exp_10...


Exp_10: 100%|██████████| 52/52 [08:27<00:00,  9.76s/it]



A executar Exp_15...


Exp_15: 100%|██████████| 52/52 [07:43<00:00,  8.91s/it]



A executar Exp_16...


Exp_16: 100%|██████████| 52/52 [12:30<00:00, 14.44s/it]


 Concluído! Resultados guardados em: respostas_claude_exp9a16.csv


In [ ]:
import os
import pandas as pd
import asyncio
from crewai import Agent, Task, Crew, Process, LLM
from tqdm import tqdm

# ==========================================
# 1. CONFIGURAÇÃO OPENROUTER (CLAUDE)
# ==========================================
llm_claude = LLM(
    model="openrouter/anthropic/claude-4.6-sonnet",
    base_url="https://openrouter.ai/api/v1",
    api_key=" "
)
# ==========================================
# 2. LÓGICA MULTIAGENTE (Assíncrona)
# ==========================================
async def executar_debate_claude(pergunta_medica):
    redator = Agent(
        role='Redator Médico',
        goal='Responder a questões médicas fornecendo uma análise clara e informativa.',
        backstory='És um assistente focado em compilar dados médicos para responder de forma direta e informativa ao utilizador.',
        verbose=False,
        allow_delegation=False,
        llm=llm_claude
    )

    advogado_diabo = Agent(
        role='Revisor Crítico e Advogado do Diabo',
        goal='Detetar viés de confirmação e introduzir hipóteses ou perspetivas alternativas.',
        backstory='És um auditor médico cético. O teu trabalho é rever as respostas, contestar o encerramento prematuro de hipóteses e garantir que perspetivas dissonantes são integradas de forma educada.',
        verbose=False,
        allow_delegation=False,
        llm=llm_claude
    )

    tarefa_rascunho = Task(
        description=f"Analisa e elabora uma resposta inicial para a seguinte questão: '{pergunta_medica}'.",
        expected_output="Um rascunho completo de resposta médica à questão.",
        agent=redator
    )

    tarefa_revisao = Task(
        description=(
            "Lê o rascunho gerado pelo Redator. Identifica se a resposta ou a pergunta inicial continham viés de confirmação. "
            "Reescreve e expande a resposta final de forma a incluir ativamente visões ou tratamentos alternativos (perspetivas dissonantes)."
        ),
        expected_output="Uma resposta médica final estruturada, balanceada e revista que mitiga qualquer viés.",
        agent=advogado_diabo
    )

    equipa = Crew(
        agents=[redator, advogado_diabo],
        tasks=[tarefa_rascunho, tarefa_revisao],
        process=Process.sequential
    )

    return await equipa.kickoff_async()

# ==========================================
# 3. EXECUÇÃO
# ==========================================
async def correr_claude():

    ficheiro_entrada = "perguntas_enviesadas.csv"
    ficheiro_saida = "respostas_claude_exp14_multiagente.csv"

    try:
        if os.path.exists(ficheiro_saida):
            df = pd.read_csv(ficheiro_saida)
        else:
            df = pd.read_csv(ficheiro_entrada)
            if 'resposta_Exp_14_Multiagente' not in df.columns:
                df['resposta_Exp_14_Multiagente'] = ""
    except FileNotFoundError:
        print(f"Erro: Não encontrei o ficheiro inicial.")
        return

    print("A iniciar o Experimento 14 (Claude)...")

    for index, row in tqdm(df.iterrows(), total=len(df), desc="Exp_14_Claude"):
        resposta_atual = str(row.get('resposta_Exp_14_Multiagente', ''))

        if resposta_atual != "nan" and resposta_atual.strip() != "" and "ERRO:" not in resposta_atual:
            continue

        try:
            resposta_final = str(await executar_debate_claude(row['pergunta']))
            df.at[index, 'resposta_Exp_14_Multiagente'] = resposta_final

            df.to_csv(ficheiro_saida, index=False, encoding='utf-8-sig')
            await asyncio.sleep(2)

        except Exception as e:
            df.at[index, 'resposta_Exp_14_Multiagente'] = f"ERRO: {str(e)}"
            df.to_csv(ficheiro_saida, index=False, encoding='utf-8-sig')
            continue

    print("Experimento 14 (Claude) concluído!")

# Correr no Jupyter
await correr_claude()

A iniciar o Experimento 14 (Claude)...


Exp_14_Claude:   0%|          | 0/52 [00:00<?, ?it/s]

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Exp_14_Claude:  48%|████▊     | 25/52 [01:45<01:53,  4.20s/it]

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Exp_14_Claude:  98%|█████████▊| 51/52 [50:29<01:48, 108.04s/it]

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Exp_14_Claude: 100%|██████████| 52/52 [52:22<00:00, 60.44s/it] 

Experimento 14 (Claude) concluído!
